# Bölüm 1 · İstatistiksel Düşünme: Sorudan Kanıta

[Atölye](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b01/) · [Veri ve lisans](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b01/VERI.md) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b01/GOREVLER.md) · [Raporlama](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b01/RAPORLAMA.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b01/COZUMLER.md)

**Başlangıç:** Google hesabınızla açın, **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU yeterlidir. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk kurulum internet ve birkaç dakika gerektirebilir. Özellikle bootstrap ve yapısal modeller daha uzun sürebilir.

Bu defter, aynı GitHub deposundaki ana Python analizini ve bölümün ileri eklerini çalıştırır. Dosya yüklemeniz veya önce başka bölümü elle yürütmeniz gerekmez. Gerçek veri, öğretim alt kümesi ve simülasyon ayrımı veri rehberinde açıklanır; bunları aynı örneklem türü saymayın. R/SPSS/Amos burada çalıştırılmaz.

## Üç çekirdek kazanım ve ileri rota

1. Araştırma hedefini, eldeki kayıtları ve analiz birimini ayırır.
2. Ölçüm türüne uygun payda ve özet seçer.
3. Verinin desteklediği kapsamda bir sonuç cümlesi yazar.

**İleri rota:** Tasarım temelli belirsizlik ve örnekleme simülasyonları; Bölüm 4 sonrasında dönülebilir.

**Ç1 · Çıkış sorusu:** Kaynak sırasındaki ilk 80 GP kaydının ortalaması Türkiye öğrencilerinin ortalaması sayılabilir mi?

Yanıtınızı çalışma sonunda öğrenci karar günlüğüne yazın. [Rubrik ve teslim rehberi](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b01/RUBRIK.md).

## 1 · Kaynakları hazırla

Sabit bir GitHub sürümündeki öğrenci paketi SHA256 ile doğrulanır. Her çalıştırma ayrı bir geçici çalışma kopyası açar; önceki sonuçlarınız silinmez. Kod ve veri, bu doğrulanan sürümden gelir. Oturum kapanmadan sonuçları indirin.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, urllib.request, zipfile, io, tempfile, subprocess, importlib.metadata
DATA_COMMIT = '23acf4be23823aa139523c8f4db34cff248c285d'
ARCHIVE_SHA256 = 'ce6440f5ee3b4dde9709e1fb250e7b7d8ac86be101b6ababacdab240443560f6'
ARCHIVE_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/ogrenci-paketi-22-bolum.zip"
cache = Path.cwd()/"kitap-colab-kaynak.zip"
if cache.exists():
    raw = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError("Kaynak indirilemedi. İnterneti kontrol edip hücreyi yeniden çalıştırın: "+ARCHIVE_URL) from exc
if hashlib.sha256(raw).hexdigest()!=ARCHIVE_SHA256:
    raise ValueError("Öğrenci paketi SHA256 denetimi başarısız. Kontrolü kaldırmayın; yerel kitap-colab-kaynak.zip dosyasını doğrulayın.")
cache.write_bytes(raw)
work = Path(tempfile.mkdtemp(prefix="b01-atolye-",dir=Path.cwd()))
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(z.infolist())<10000 and sum(i.file_size for i in z.infolist())<134217728
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if work.resolve() not in target.parents or ((member.external_attr>>16)&0o170000)==0o120000:
            raise ValueError("Güvenli olmayan arşiv yolu")
    z.extractall(work)
root = work/"ileri-istatistik-22-bolum"
assert (root/"BOLUM_ESLESMESI.json").is_file()
application=root/'uygulamalar/b01-istatistiksel-mekanizma'
modules=['uygulamalar/b01-istatistiksel-mekanizma/tasarim-belirsizligi']
expected_main=json.loads((root/"bolumler/b01/beklenen.json").read_text())
expected_extra={m:json.loads((root/m/"sonuclar/ozet.json").read_text()) for m in modules}
print("Kaynak doğrulandı; Bölüm 1 çalışma kopyası hazır.")

## 2 · Analiz ortamını hazırla

Sınanmış paket sürümleri kullanılır. Mevcut çekirdek bu sürümlerdeyse doğrudan kullanılır. Farklıysa analizler ayrı bir Python ortamında kurulur; Colab çekirdeğindeki paketler topluca değiştirilmez. Kurulum başarısız olursa hücredeki hatayı inceleyin. Python 3.12 veya üzeri çalışma zamanı önerilir. GPU gerekmez.

In [ ]:
FORCE_ISOLATED=globals().get("FORCE_ISOLATED",False)
PINS={'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0', 'matplotlib': '3.11.2', 'statsmodels': '0.15.0', 'PyMuPDF': '1.26.6'}
def package_versions(executable):
    check="import importlib.metadata,json; names="+repr(list(PINS))+";print(json.dumps({n:importlib.metadata.version(n) for n in names}))"
    attempt=subprocess.run([str(executable),"-c",check],capture_output=True,text=True)
    return json.loads(attempt.stdout) if attempt.returncode==0 else {}
python=Path(sys.executable)
if FORCE_ISOLATED or package_versions(python)!=PINS:
    environment=Path.cwd()/".kitap-colab-python"
    if not (environment/"bin/python").exists():
        subprocess.run([sys.executable,"-m","venv",str(environment)],check=True)
    python=environment/"bin/python"
    if package_versions(python)!=PINS:
        subprocess.run([str(python),"-m","pip","install",*[name+"=="+version for name,version in PINS.items()]],check=True)
assert package_versions(python)==PINS,"Paket sürümleri sınanmış ortamla eşleşmiyor"
versions=package_versions(python)
run_env={**os.environ,"MPLBACKEND":"Agg","MPLCONFIGDIR":str(work/"mpl"),"OPENBLAS_NUM_THREADS":"1","OMP_NUM_THREADS":"1","PYTHONDONTWRITEBYTECODE":"1"}
(work/"logs").mkdir(exist_ok=True)
def run_script(relative):
    print("Çalışıyor:",relative,flush=True)
    result=subprocess.run([str(python),str(root/relative)],cwd=root,env=run_env,capture_output=True,text=True,timeout=900)
    (work/"logs"/(relative.replace("/","__")+".log")).write_text(result.stdout+"\n"+result.stderr)
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+"\n"+result.stderr[-4000:])
    print("Tamamlandı.")
print("Analiz ortamı hazır:",versions)

## Ana analiz

Kaynak ve yöntem için yukarıdaki veri rehberini okuyun. Ana analiz ve sonradan eklenen duyarlılık uygulamaları ayrı çalıştırılır. Sonuçlar önceden gösterilen tabloların kopyası olarak değil, kod yeniden yürütülerek üretilir.

In [ ]:
before_figures=set((root/"assets_figures").glob("*.pdf"))
run_script("bolumler/b01/analiz.py")
main_result=json.loads((application/"sonuclar/ozet.json").read_text())
from IPython.display import display, JSON, HTML, Image
print("Ana sonuçlar — ayrıntıları açmak için okları kullanın:")
display(JSON(main_result,expanded=False))

## İleri uygulamalar ve duyarlılık

Her ekin açıklaması atölye sayfasındadır. Bu sonuçları ana analizle veya birbirinden farklı tahmin hedefleriyle karıştırmayın. Hazır kontrol betikleri varsa önce onlar, ardından ek analiz çalışır.

In [ ]:
extra_results={}
for module in modules:
    if (root/module/"dogrula.py").is_file(): run_script(module+"/dogrula.py")
    run_script(module+"/analiz.py")
    extra_results[module]=json.loads((root/module/"sonuclar/ozet.json").read_text())
    print(module)
    display(JSON(extra_results[module],expanded=False))

## Sonuçları referanslarla karşılaştır

Kitabın sayısal referansları, önceki sürümden değişmeden alınır. Sürüm/zaman gibi hesap dışı alanlar mevcut karşılaştırma politikasına göre dışlanır. Hata oluşursa veri, sürüm ve günlük dosyalarını kontrol edin; referansları sonuca uydurmayın. Bu kontrol bilimsel varsayımları veya SPSS yürütmesini doğrulamaz.

In [ ]:
sys.path.insert(0,str(root/"tools"))
from dogrula_22 import compare
checks={"main":compare(expected_main,main_result)}
for module in modules:
    checks[module]=compare(expected_extra[module],json.loads((root/module/"sonuclar/ozet.json").read_text()))
assert all(not differences for differences in checks.values()),checks
print("Ana analiz ve",len(modules),"ileri ek için sayısal referanslar eşleşti.")

## Grafikler ve özet tablolar

Grafikler bu çalıştırmada üretilir. Ön koşul olarak çalıştırılan Bölüm 18'in şekilleri burada tekrar gösterilmez. CSV tablolarının ilk 12 satırı görüntülenir; tam dosyalar çıktı paketindedir. Şekil sayısı bölüme göre değişir.

In [ ]:
import csv,html
new_figures=sorted(set((root/"assets_figures").glob("*.pdf"))-before_figures)
for figure in new_figures:
    png=figure.with_suffix(".png")
    renderer="import fitz,sys;d=fitz.open(sys.argv[1]);d[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(sys.argv[2])"
    subprocess.run([str(python),"-c",renderer,str(figure),str(png)],check=True)
    print(figure.name)
    display(Image(filename=str(png)))
for table in sorted((application/"sonuclar").glob("*.csv"))[:3]:
    with table.open(encoding="utf-8-sig",newline="") as f: rows=list(csv.reader(f))
    content="<h4>"+html.escape(table.name)+"</h4><table>"
    for i,row in enumerate(rows[:13]):
        tag="th" if i==0 else "td"
        content+="<tr>"+"".join("<"+tag+">"+html.escape(value)+"</"+tag+">" for value in row)+"</tr>"
    display(HTML(content+"</table>"))

## Sıra sizde

# Bölüm 1 — görevler

Güncel kitap bölümünün etkinlik ve alıştırmalarıdır. Önce kendi çözümünüzü
üretin; ardından çözümleri kontrol edin. Kitap sayfa numaraları PDF sürümüne
göre değişebilir.

## D1–D3: Hesaptan Önce

1.  “Türkiye’deki lise öğrencilerinin yıl sonu başarısı nedir?” sorusuna bu dosya tek başına yanıt verebilir mi?

2.  Bir satır neyi temsil eder? G1 ve G3’ü alt alta yazınca öğrenci sayısı iki katına çıkar mı?

3.  Kaynak sıra numarası ve okul koduyla ortalama hesaplamak, bir not ortalamasıyla aynı anlamı taşır mı?

# Rehberli ve Bağımsız Etkinlikler

## G1–G4: Rehberli Çözüm

1.  80 kaydı öğretim anakütlesi kabul ederek parametreyi ve basit rastgele 10 kaydın istatistiğini yazın. Aralarındaki farkı hesaplayın.

2.  $j=8$ ve $r=3$ için seçilen satırları, n’yi ve G3 ortalamasını bulun. Bu tek sonuç yanlılık kanıtı mıdır?

3.  Üç G3 bandının frekans ve yüzdelerini hesaplayın; paydanızı ve bantların statüsünü belirtin.

4.  Eksik-veri öğretim kopyasında 15–20 bandı için toplam ve geçerli yüzdeleri bulun; gerçek dosyayla karıştırmadan açıklayın.

## B1–B3: Bağımsız Çözüm

1.  Sistematik başlangıç $r=5$ olursa satırları ve ortalamayı bulun. Sekiz başlangıcın ortalaması neden 12.6375’tir?

2.  Uzun biçimdeki 160 satır neden 160 bağımsız öğrenci değildir? Tek okul kodundan okul farkı hesaplanabilir mi?

3.  Kurgusal döngü listesinde başlangıç 1 ve 3 için oranları bulun. Rastgele başlangıç ile sabit başlangıcın farkını açıklayın.

## H1: Hatalı Raporu Onarma

> “İlk 80 kaydı aldığımız için örnek rastgeledir. Not dosyasını uzatınca 160 öğrenci oldu. G1=0 kaydını eksik diye sildik. GP’nin yüzde 100 olması iki okulun tüm öğrencilerinin GP’de olduğunu gösterir. Öğretim kopyasındaki geçerli yüzde 19.74’ü tüm 80 kişinin oranı yazdık. 16 puan alanın yeteneği 8 puan alanın iki katıdır.”

Altı iddiayı birim, seçim, payda ve ölçüm düzeyi bakımından düzeltin.

## P1: Bir Sayfalık Veri Tanıma Dosyası

Yeni bir araştırma için hedefi yer/zamanla tanımlayın; çerçeve, seçim, birim, veri sözlüğü, eksikler, uygun grafik ve genelleme sınırını yazın. Yanıt vermeyenleri ve çerçeve dışında kalanları ayırın. Henüz veri toplanmadıysa planı gerçekleşmiş çalışma gibi sunmayın. Rubrik: soru/kapsam 25, birim/ölçüm 25, seçim/payda 25, grafik/rapor 15, kaynak/yeniden üretim 10 puan.

Yanıtlar Ek F, Bölüm 1’dedir (s. kitaptaki ilgili sayfa).

# Alıştırmalar

Bu alıştırmalar, istatistiksel mekanizmanın temel kavramlarını veri türü, ölçüm düzeyi, örnekleme ve grafik seçimi bağlamında pekiştirmek için üç düzeyde düzenlenmiştir. Yanıtlar ve kısa çözüm ipuçları Ek F’dedir.

## Kolay Düzey

1.  Parametre ile istatistik arasındaki temel farkı bir cümleyle açıklayınız.

2.  Nominal, ordinal, aralık ve oran ölçüm düzeylerine birer örnek veriniz.

3.  Kategorik bir değişken için çubuk grafik mi histogram mı daha uygundur? Gerekçenizi yazınız.

4.  Sistematik örneklemede anakütle büyüklüğü $N=1200$ ve örneklem büyüklüğü $n=100$ ise seçim aralığını hesaplayınız.

5.  Bir araştırmada örneklemin anakütleyi temsil etmemesi hangi temel soruna yol açar?

## Orta Düzey

1.  500 öğrencilik bir fakülteden 60 öğrenci seçilerek ortalama sınav kaygısı ölçülüyor. Anakütle, örneklem, parametre ve istatistik kavramlarını bu bağlamda belirtiniz.

2.  Yaş, cinsiyet, eğitim düzeyi, aylık gelir ve memnuniyet puanı değişkenlerini veri türü ve ölçüm düzeyi açısından sınıflandırınız.

3.  Bir veri kümesinde sayısal değişken çok sağa çarpık ise yalnız aritmetik ortalama raporlamak neden yanıltıcı olabilir?

4.  Kolayda örnekleme ile toplanan bir çevrimiçi anketin genellenebilirliği neden sınırlıdır?

5.  Aynı veriyi farklı eksen ölçekleriyle çizmenin yorum üzerindeki etkisini açıklayınız.

## İleri Düzey

1.  Bir araştırmacı ordinal Likert maddelerinin ortalamasını alıp parametrik test uygulamak istiyor. Bu kararın hangi koşullarda savunulabileceğini, hangi koşullarda sorunlu olabileceğini tartışınız.

2.  Tabakalı örnekleme ile küme örnekleme arasındaki farkı, eğitim araştırmasından birer örnekle açıklayınız.

3.  Eksik verinin tamamen rastgele oluşmaması, istatistiksel mekanizmanın hangi aşamalarında yanlılık doğurabilir?

4.  Betimsel grafiklerde çarpıklık, aykırı değer ve grup farkı aynı anda görülmek isteniyorsa hangi grafik kombinasyonları tercih edilebilir?

5.  Bir araştırma raporunda veri türü, ölçüm düzeyi, örnekleme yöntemi ve grafik seçiminin birlikte raporlanması neden bilimsel denetlenebilirliği artırır?

Yöntem ve kaynak belgelerine erişim: 6 Eylül 2026.


## Revizyon 5: yeni hedef ve tasarım görevleri

## T1–T3: Hedef ve Tasarım Denetimi

1.  Kapalı 80 kayıt için ortalama ile 15 puan eşiğini karşılayanların oranını hedef parametre olarak yazın. 649 kaydın ortalaması neden bu iki sayıdan belirlenemez? Altı satırlık analiz sözleşmesini tamamlayın.

2.  $N=80$, $n=10$, $S_{80}^{2}=4.107437$ ile basit rastgele seçimin standart hatasını hesaplayın. Sekiz sistematik ortalamanın tam tasarım varyansını neden 8’e bölerek buluruz? Sistematik yöntemin her zaman üstün olduğunu söyleyebilir miyiz?

3.  Araştırmacı “20 000 tekrar sayesinde 200 000 bağımsız öğrenciye ulaştım; Türkiye ortalamasını çok kesin buldum” diyor. Veri büyüklüğü, Monte Carlo hatası ve genelleme bakımından bu cümleyi düzeltin.


Önce kendi yorumunuzu yazın; sonra [çözümlerle](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b01/COZUMLER.md) karşılaştırın. Sonuç, belirsizlik, varsayım ve sınırlılıkları [raporlama rehberine](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b01/RAPORLAMA.md) göre ayırın.

In [ ]:
yanitlarim={"bulgular":"", "belirsizlik":"", "varsayimlar":"", "sinirliliklar":""}
# Metinleri doldurun; sayısal kontrolü geçmek bu yorumları tamamlamaz.

## Çıktıları indir

Son hücre sonuç JSON/CSV'lerini, grafiklerin PDF/PNG dosyalarını, günlükleri ve kendi yanıtlarınızı paketler. Colab'ın solundaki Dosyalar panelini yenileyin ve aşağıda yazan ZIP'i indirin. Geçici çalışma zamanı kapanmadan bunu yapın; kod Drive'a otomatik erişmez.

## Öğrenci kararı ve kısa savunma

**Karar görevi:** Betimleme hedefini ilk 80 kayıt veya ilk 40 kayıt olarak seçin; iki özeti karşılaştırın ve hangi hedefe genellemediğinizi yazın.

Yalnız bir kararı değiştirin. Ana sonucu koruyun; gerekirse yeni bir hesap hücresi ekleyip denemeyi ayrı kaydedin. Sayısal referansları yeni sonuca uydurmayın. Önce/sonra kanıtını sonuç dosyası, tablo veya açık bir hesapla ilişkilendirin. Yapılmamış hesabı yapılmış gibi yazmayın.

**3–5 dakikalık savunma:** Hangi hedefi seçtiniz ve neden? Tam olarak ne değişti? Hangi belirsizlik ve kapsam sınırı kaldı? Karşılaştırma aynı hipotezi mi sınadı?

Aşağıdaki alanları kendi çalışmanızla doldurun. **Tümünü çalıştır**, bu alanları sizin yerinize tamamlamaz. Sayısal kontrollerin geçmesi öğrenci kararının tamamlandığı anlamına gelmez. Alanların dolu olması da kanıtın doğruluğunu otomatik onaylamaz.

In [ ]:
karar_gunlugu = {
    "bolum": 1,
    "gorev": 'Betimleme hedefini ilk 80 kayıt veya ilk 40 kayıt olarak seçin; iki özeti karşılaştırın ve hangi hedefe genellemediğinizi yazın.',
    "kararim": "",
    "gerekcem": "",
    "degistirdigim_tek_sey": "",
    "onceki_kanit": "",
    "sonraki_kanit": "",
    "belirsizlik_ve_sinir": "",
    "kisa_savunma": "",
    "cikis_yanitim": ""
}
zorunlu_alanlar = ["kararim","gerekcem","degistirdigim_tek_sey","onceki_kanit","sonraki_kanit","belirsizlik_ve_sinir","kisa_savunma","cikis_yanitim"]
karar_gunlugu["alanlar_dolu"] = all(isinstance(karar_gunlugu[k],str) and bool(karar_gunlugu[k].strip()) for k in zorunlu_alanlar)
karar_gunlugu["durum"] = "Öğretim elemanı incelemesi gerekir" if karar_gunlugu["alanlar_dolu"] else "Öğrenci kararı henüz tamamlanmadı"
print(karar_gunlugu["durum"])
# Bu hücreyi doldurduktan sonra çıktı ZIP'i hücresini yeniden çalıştırın.

In [ ]:
manifest={"chapter":1,"source_commit":DATA_COMMIT,"source_archive_sha256":ARCHIVE_SHA256,"versions":versions,"checks":checks,"answers":yanitlarim,"scope":"Python ana analiz ve ileri ekler; SPSS/R/Amos çalıştırılmadı"}
# OGRETIM_EXPORT_BEGIN
manifest["ogrenci_karari"] = karar_gunlugu
(work/"ogrenci-karari.json").write_text(json.dumps(karar_gunlugu,ensure_ascii=False,indent=2),encoding="utf-8")
# OGRETIM_EXPORT_END
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive=Path.cwd()/"b01-colab-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(work/"ogrenci-karari.json","ogrenci-karari.json")
    z.write(work/"calisma-kaydi.json","calisma-kaydi.json")
    for module in ['uygulamalar/b01-istatistiksel-mekanizma',*modules]:
        for f in sorted((root/module/"sonuclar").rglob("*")):
            if f.is_file() and f.suffix in [".json",".csv"]:z.write(f,str(f.relative_to(root)))
    for figure in new_figures:
        for f in [figure,figure.with_suffix(".png")]: z.write(f,"grafikler/"+f.name)
    for f in (work/"logs").glob("*.log"):z.write(f,"logs/"+f.name)
print("İndirmeye hazır:",archive.name)